# Classificação de Logs de Firewall com Programação Genética

**Disciplina:** SCC0370 – Introdução à Computação Bioinspirada  
**Semestre:** Primeiro Semestre de 2026  

## Reprodução

### Google Colab
1. Faça upload deste notebook e dos dois CSVs para o Colab (ou monte o Google Drive).
2. Ajuste `DATA_DIR` na célula de configuração para o caminho correto.
3. Execute todas as células em ordem (Runtime → Run all).

### Local
```bash
pip install deap numpy pandas scikit-learn matplotlib
jupyter notebook classificacao_logs_firewall.ipynb
```

### Dependências
| Pacote | Versão mínima |
|---|---|
| deap | 1.4.1 |
| numpy | 1.26.4 |
| pandas | 2.2.2 |
| scikit-learn | 1.5.2 |
| matplotlib | 3.9.2 |

---
## 1. Setup & Reprodutibilidade

In [26]:
# Instala dependências (necessário no Colab)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deap"], check=True)
print("Dependências OK")

Dependências OK


You should consider upgrading via the '/Users/renatospessotto/Documents/USP/2026_Primeiro_Semestre/bioinspirada/Projeto/.venv/bin/python -m pip install --upgrade pip' command.


In [27]:
import os, random, warnings, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend sem janela (Colab/nbconvert)
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from deap import base, creator, gp, tools, algorithms
import operator

warnings.filterwarnings('ignore')

# ── Semente global ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Configuração central ─────────────────────────────────────────────────────
# Ajuste os hiperparâmetros aqui; o resto do notebook usa este dict.
CFG = dict(
    # Caminhos
    DATA_DIR   = "data",                                          # <-- altere se necessário
    FILE_DS1   = "Log-Completo-Todos-Atributos-Importantes.csv",
    FILE_DS2   = "log-completo-Todos-Atributos-Importantes-fase-online-2.csv",

    # Split
    TRAIN_FRAC = 0.30,   # 30% de cada dataset (DS1 e DS2) → treino; 70% → teste
    N_WINDOWS  = 50,     # janelas de avaliação no teste

    # GP
    POP_SIZE   = 200,    # tamanho da população por classe (4 evoluções independentes)
    N_GEN      = 40,     # gerações por evolução
    CXPB       = 0.7,    # probabilidade de crossover
    MUTPB      = 0.3,    # probabilidade de mutação
    TOURNSIZE  = 5,      # tamanho do torneio de seleção
    MAX_HEIGHT = 6,      # altura máxima da árvore (controle de bloat)
    HOF_SIZE   = 1,      # melhores indivíduos guardados por HallOfFame

    # Múltiplas execuções para estatísticas de robustez
    N_RUNS     = 1,      # aumente para 3-5 se tiver tempo (Colab ~20-30min/run)
)

print("Configuração carregada:", CFG)

Configuração carregada: {'DATA_DIR': 'data', 'FILE_DS1': 'Log-Completo-Todos-Atributos-Importantes.csv', 'FILE_DS2': 'log-completo-Todos-Atributos-Importantes-fase-online-2.csv', 'TRAIN_FRAC': 0.3, 'N_WINDOWS': 50, 'POP_SIZE': 200, 'N_GEN': 40, 'CXPB': 0.7, 'MUTPB': 0.3, 'TOURNSIZE': 5, 'MAX_HEIGHT': 6, 'HOF_SIZE': 1, 'N_RUNS': 1}


---
## 2. Carga e Montagem dos Dados

Os dois conjuntos são concatenados **em ordem cronológica** (DS1 → DS2), formando um único fluxo de ~1,5 milhão de instâncias.

> **Suposição temporal:** a ordem das linhas dentro de cada arquivo é assumida como a ordem de captura das sessões. Não há coluna de timestamp explícita; o contexto do trabalho confirma a coleta sequencial por dia.

In [28]:
def load_stream(cfg):
    path1 = os.path.join(cfg['DATA_DIR'], cfg['FILE_DS1'])
    path2 = os.path.join(cfg['DATA_DIR'], cfg['FILE_DS2'])

    print(f"Carregando DS1: {path1}")
    df1 = pd.read_csv(path1, low_memory=False)
    df1['phase'] = 'inicial'

    print(f"Carregando DS2: {path2}")
    df2 = pd.read_csv(path2, low_memory=False)
    df2['phase'] = 'online'

    # Remove linhas sem rótulo antes de concatenar para evitar classe NaN
    df1 = df1.dropna(subset=['Action'])
    df2 = df2.dropna(subset=['Action'])

    df = pd.concat([df1, df2], ignore_index=True)
    print(f"\nFluxo total: {len(df):,} instâncias | {df.shape[1]} colunas")
    print(f"  DS1: {len(df1):,}  |  DS2: {len(df2):,}")
    print(f"  Fronteira de concept drift: índice {len(df1):,}")
    return df, len(df1)

df_raw, DRIFT_IDX = load_stream(CFG)
print("\nDistribuição da classe alvo (fluxo completo):")
print(df_raw['Action'].value_counts().to_string())
print("\n% relativa:")
print((df_raw['Action'].value_counts(normalize=True) * 100).round(2).to_string())

Carregando DS1: data/Log-Completo-Todos-Atributos-Importantes.csv
Carregando DS2: data/log-completo-Todos-Atributos-Importantes-fase-online-2.csv

Fluxo total: 1,517,735 instâncias | 28 colunas
  DS1: 1,002,160  |  DS2: 515,575
  Fronteira de concept drift: índice 1,002,160

Distribuição da classe alvo (fluxo completo):
Action
allow         818352
deny          593736
drop          103187
reset-both      2460

% relativa:
Action
allow         53.92
deny          39.12
drop           6.80
reset-both     0.16


---
## 3. Análise Exploratória (EDA)

In [29]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribuição de classes
counts = df_raw['Action'].value_counts()
bars = axes[0].bar(counts.index, counts.values, color=['#2196F3','#F44336','#FF9800','#9C27B0'])
axes[0].set_title('Distribuição da Classe Alvo (fluxo completo)', fontsize=12)
axes[0].set_ylabel('Contagem')
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 2000,
                 f'{b.get_height():,}', ha='center', va='bottom', fontsize=9)

# Mapa de ausências (% de NaN por coluna)
miss = (df_raw.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
axes[1].barh(miss.index, miss.values, color='#EF5350')
axes[1].set_xlabel('% NaN')
axes[1].set_title('Colunas com Valores Ausentes', fontsize=12)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())

plt.tight_layout()
plt.savefig('figures/fig_eda_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()

In [30]:
# Cardinalidade das colunas categóricas e problemáticas
cat_cols = df_raw.select_dtypes('object').columns.tolist()
card = df_raw[cat_cols].nunique().sort_values(ascending=False)
print("Cardinalidade das colunas categóricas:")
print(card.to_string())

print("\nColunas numéricas:")
print(df_raw.select_dtypes('number').describe().T[['mean','std','min','max']].to_string())

Cardinalidade das colunas categóricas:
Source address         91350
Destination address    57064
NAT Destination IP      4113
NAT Source IP           1109
Application              285
Source Country           204
Destination Country      171
Container of app          80
Category                  65
Subcategory of app        35
Source Zone               20
Destination Zone          20
Category of app            7
IP Protocol                6
Technology of app          5
Action                     4
phase                      2

Colunas numéricas:
                              mean           std  min           max
Source Port           41233.183127  1.801055e+04  0.0  6.553500e+04
Destination Port       7258.794604  1.494590e+04  0.0  6.553500e+04
NAT Source Port        1060.240071  6.570494e+03  0.0  6.551400e+04
NAT Destination Port     19.364517  5.948471e+02  0.0  3.024700e+04
Bytes                 23205.392449  5.640966e+06  0.0  3.294836e+09
Bytes Sent             1545.306921  1.71

---
## 4. Pré-processamento

**Decisões:**
- **Descartadas:** `Source address`, `Destination address` (91k/57k valores únicos – quasi-identificadores), `NAT Source IP`, `NAT Destination IP` (97% ausentes), `Container of app` (~75% ausentes), `phase` (variável de controle interna).
- **Categóricas** (cardinalidade gerenciável): *frequency encoding* – cada valor é substituído pela sua frequência relativa no treino. Mantém o conjunto de terminais do GP pequeno e todas as colunas numéricas.
- **Numéricas de cauda pesada** (`Bytes`, `Bytes Sent/Received`, `Packets`, `Elapsed Time`): `log1p` antes da padronização.
- **Padronização z-score** (scaler fitado **apenas no treino** para evitar vazamento).
- **NaN** numéricos → mediana do treino.

> **Anti-vazamento:** todos os encoders e o scaler são fitados exclusivamente nos 30% de treino e depois aplicados ao teste.

In [31]:
# Colunas a descartar
DROP_COLS = [
    'Source address', 'Destination address',
    'NAT Source IP', 'NAT Destination IP',
    'Container of app', 'phase',
]

# Colunas categóricas restantes (após descarte e sem 'Action')
CAT_COLS = [
    'Application', 'Source Zone', 'Destination Zone',
    'IP Protocol', 'Category',
    'Source Country', 'Destination Country',
    'Subcategory of app', 'Category of app', 'Technology of app',
]

# Colunas numéricas com cauda pesada → log1p
LOG_COLS = [
    'Bytes', 'Bytes Sent', 'Bytes Received',
    'Packets', 'Packets Sent', 'Packets Received',
    'Elapsed Time (sec)',
]

# Portas e NAT portas — numéricas mas não log1p
PORT_COLS = [
    'Source Port', 'Destination Port',
    'NAT Source Port', 'NAT Destination Port',
]

FEATURE_COLS = CAT_COLS + LOG_COLS + PORT_COLS  # ordem fixa
print(f"Features: {len(FEATURE_COLS)} colunas → {FEATURE_COLS}")

Features: 21 colunas → ['Application', 'Source Zone', 'Destination Zone', 'IP Protocol', 'Category', 'Source Country', 'Destination Country', 'Subcategory of app', 'Category of app', 'Technology of app', 'Bytes', 'Bytes Sent', 'Bytes Received', 'Packets', 'Packets Sent', 'Packets Received', 'Elapsed Time (sec)', 'Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port']


In [32]:
# Encoder de classes
label_enc = LabelEncoder()
y_all = label_enc.fit_transform(df_raw['Action'])
CLASS_NAMES = list(label_enc.classes_)
N_CLASSES = len(CLASS_NAMES)
print(f"Classes: {CLASS_NAMES}  (índices 0..{N_CLASSES-1})")

# Split proporcional por dataset: 30% de cada DS → treino, 70% de cada DS → teste
# Garante que o modelo vê exemplos dos dois períodos durante o treino.
n_ds1 = DRIFT_IDX
n_ds2 = len(df_raw) - DRIFT_IDX

n_train_ds1 = int(n_ds1 * CFG['TRAIN_FRAC'])
n_train_ds2 = int(n_ds2 * CFG['TRAIN_FRAC'])

train_idx = np.concatenate([
    np.arange(n_train_ds1),                        # primeiros 30% do DS1
    np.arange(n_ds1, n_ds1 + n_train_ds2),         # primeiros 30% do DS2
])
test_idx = np.concatenate([
    np.arange(n_train_ds1, n_ds1),                 # últimos 70% do DS1
    np.arange(n_ds1 + n_train_ds2, len(df_raw)),   # últimos 70% do DS2
])

n_train = len(train_idx)
n_total = len(df_raw)

# Posição no array de teste onde os exemplos de DS2 começam
drift_test_idx = n_ds1 - n_train_ds1

print(f"\nDS1: {n_ds1:,}  |  DS2: {n_ds2:,}")
print(f"Treino: {n_train:,} ({n_train/n_total*100:.1f}%)  |  Teste: {len(test_idx):,} ({len(test_idx)/n_total*100:.1f}%)")
print(f"  Treino DS1: {n_train_ds1:,}  |  Treino DS2: {n_train_ds2:,}")
print(f"  Teste  DS1: {n_ds1-n_train_ds1:,}  |  Teste  DS2: {n_ds2-n_train_ds2:,}")
print(f"  Fronteira DS1→DS2 no teste: posição {drift_test_idx:,}")

Classes: ['allow', 'deny', 'drop', 'reset-both']  (índices 0..3)

DS1: 1,002,160  |  DS2: 515,575
Treino: 455,320 (30.0%)  |  Teste: 1,062,415 (70.0%)
  Treino DS1: 300,648  |  Treino DS2: 154,672
  Teste  DS1: 701,512  |  Teste  DS2: 360,903
  Fronteira DS1→DS2 no teste: posição 701,512


In [33]:
def build_features(df_full, train_idx, cat_cols, log_cols, port_cols):
    """Preprocessa o fluxo completo fitando encoders/scaler apenas no treino."""
    df = df_full.copy()

    train_slice = df.iloc[train_idx]

    # ── Frequency encoding para categóricas ────────────────────────────────
    freq_maps = {}
    for col in cat_cols:
        freq = train_slice[col].value_counts(normalize=True)
        freq_maps[col] = freq
        df[col] = df[col].map(freq).fillna(0.0)

    # ── log1p nas numéricas de cauda pesada ────────────────────────────────
    for col in log_cols:
        df[col] = np.log1p(df[col].fillna(0).clip(lower=0))

    # ── Portas: NaN → 0 ───────────────────────────────────────────────────
    for col in port_cols:
        df[col] = df[col].fillna(0.0)

    all_num_cols = cat_cols + log_cols + port_cols
    X = df[all_num_cols].values.astype(np.float32)

    # ── NaN remanescentes → mediana do treino ──────────────────────────────
    train_medians = np.nanmedian(X[train_idx], axis=0)
    for j in range(X.shape[1]):
        mask = np.isnan(X[:, j])
        X[mask, j] = train_medians[j]

    # ── Padronização z-score (fitada só no treino) ─────────────────────────
    scaler = StandardScaler()
    scaler.fit(X[train_idx])
    X = scaler.transform(X).astype(np.float32)

    assert not np.isnan(X).any(), "Ainda há NaN após pré-processamento!"
    print(f"X shape: {X.shape}  dtype: {X.dtype}  NaN: {np.isnan(X).sum()}")
    return X, scaler, freq_maps, all_num_cols


X_all, scaler, freq_maps, FEAT_NAMES = build_features(
    df_raw, train_idx, CAT_COLS, LOG_COLS, PORT_COLS
)
y_all = y_all.astype(np.int32)

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_test,  y_test  = X_all[test_idx],  y_all[test_idx]

print(f"\nX_train: {X_train.shape}  X_test: {X_test.shape}")
print("Distribuição de classes no treino:",
      {CLASS_NAMES[i]: int((y_train==i).sum()) for i in range(N_CLASSES)})

X shape: (1517735, 21)  dtype: float32  NaN: 0

X_train: (455320, 21)  X_test: (1062415, 21)
Distribuição de classes no treino: {'allow': 259273, 'deny': 166496, 'drop': 28739, 'reset-both': 812}


---
## 5. Modelo GP (DEAP) — One-vs-Rest

### Estratégia
Evoluímos **4 árvores independentes**, uma por classe (OvR). Cada árvore recebe os 21 atributos numéricos e retorna um *score* escalar. A predição final é `argmax` dos 4 scores.

**Fitness de cada árvore:** F1 binário (classe k vs. resto) no treino completo, avaliado de forma **vetorizada em NumPy** (toda a matriz de treino numa única operação) para viabilizar o uso do treino completo (~450k linhas).

**Primitivas:**  `+`, `-`, `*`, divisão protegida, `neg`, `abs`, `max2`, `min2`  
**Terminais:**  um terminal por atributo (`x0`…`x_n`) + constantes efêmeras (`float` em [-1, 1])

In [34]:
N_FEATURES = X_train.shape[1]

# ── Primitivas ──────────────────────────────────────────────────────────────
def protected_div(a, b):
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(np.abs(b) < 1e-7, np.ones_like(a), a / b)

def safe_neg(a):  return -a
def safe_abs(a):  return np.abs(a)
def safe_max(a, b): return np.maximum(a, b)
def safe_min(a, b): return np.minimum(a, b)

def build_pset(n_features, feat_names):
    pset = gp.PrimitiveSet("MAIN", n_features)
    # Renomeia terminais para os nomes reais dos atributos
    for i, name in enumerate(feat_names):
        safe_name = 'x' + str(i)  # DEAP exige nomes de argN
        pset.renameArguments(**{f'ARG{i}': safe_name})

    pset.addPrimitive(operator.add, 2, name='add')
    pset.addPrimitive(operator.sub, 2, name='sub')
    pset.addPrimitive(operator.mul, 2, name='mul')
    pset.addPrimitive(protected_div, 2, name='div')
    pset.addPrimitive(safe_neg, 1, name='neg')
    pset.addPrimitive(safe_abs, 1, name='abs')
    pset.addPrimitive(safe_max, 2, name='max2')
    pset.addPrimitive(safe_min, 2, name='min2')

    pset.addEphemeralConstant('rand_const', lambda: random.uniform(-1, 1))
    return pset


pset = build_pset(N_FEATURES, FEAT_NAMES)
print(f"Primitivos: {[p.name for p in pset.primitives[object]]}")
print(f"Terminais (atributos + ephemeral): {N_FEATURES} atributos")

Primitivos: ['add', 'sub', 'mul', 'div', 'neg', 'abs', 'max2', 'min2']
Terminais (atributos + ephemeral): 21 atributos


In [35]:
# ── Tipos DEAP ──────────────────────────────────────────────────────────────
# Criados uma única vez; blocos try/except evitam erro em re-execução de célula
try:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
except Exception:
    pass
try:
    creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)
except Exception:
    pass


def build_toolbox(pset, cfg):
    tb = base.Toolbox()
    tb.register("expr",   gp.genHalfAndHalf, pset=pset, min_=1, max_=4)
    tb.register("individual", tools.initIterate, creator.Individual, tb.expr)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("compile",    gp.compile, pset=pset)
    tb.register("select",  tools.selTournament, tournsize=cfg['TOURNSIZE'])
    tb.register("mate",    gp.cxOnePoint)
    tb.register("expr_mut", gp.genFull, min_=0, max_=2)
    tb.register("mutate",  gp.mutUniform, expr=tb.expr_mut, pset=pset)
    # Controle de bloat
    tb.decorate("mate",   gp.staticLimit(key=operator.attrgetter("height"), max_value=cfg['MAX_HEIGHT']))
    tb.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=cfg['MAX_HEIGHT']))
    return tb


toolbox = build_toolbox(pset, CFG)
print("Toolbox DEAP configurado.")

Toolbox DEAP configurado.


In [36]:
# ── Função de fitness vetorizada ────────────────────────────────────────────
def make_eval_func(X_tr, y_tr, class_idx, toolbox):
    """
    Retorna uma função de avaliação para a classe `class_idx` (OvR).
    Avalia cada indivíduo no treino completo usando NumPy vetorizado.
    """
    y_binary = (y_tr == class_idx).astype(np.int32)
    # Pré-calcula colunas como tuplas para passar à função compilada
    cols = [X_tr[:, j] for j in range(X_tr.shape[1])]

    def evaluate(individual):
        func = toolbox.compile(expr=individual)
        try:
            scores = func(*cols)  # vetorizado: retorna array de shape (n_train,)
            # np.isscalar não captura arrays 0-dim (ex.: np.int32(1)) em NumPy ≥ 2.0
            # np.atleast_1d + broadcast garante shape (n_train,) em todos os casos
            scores = np.broadcast_to(
                np.atleast_1d(np.asarray(scores, dtype=np.float64)),
                (len(y_binary),)
            ).copy()
            preds = (scores > 0).astype(np.int32)
        except Exception:
            return (0.0,)
        f1 = f1_score(y_binary, preds, zero_division=0)
        return (f1,)

    return evaluate


print("Função de fitness vetorizada definida.")

Função de fitness vetorizada definida.


In [37]:
# ── Evolução de 1 árvore (1 classe) ────────────────────────────────────────
def evolve_one_class(class_idx, X_tr, y_tr, toolbox, cfg, seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    cls_name = CLASS_NAMES[class_idx]
    eval_fn = make_eval_func(X_tr, y_tr, class_idx, toolbox)
    toolbox.register("evaluate", eval_fn)

    pop  = toolbox.population(n=cfg['POP_SIZE'])
    hof  = tools.HallOfFame(cfg['HOF_SIZE'])

    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("max",  np.max)
    stats.register("mean", np.mean)

    print(f"  [Classe '{cls_name}'] Evoluindo {cfg['POP_SIZE']} indivíduos × {cfg['N_GEN']} gerações...")
    t0 = time.time()
    pop, log = algorithms.eaSimple(
        pop, toolbox,
        cxpb=cfg['CXPB'], mutpb=cfg['MUTPB'],
        ngen=cfg['N_GEN'],
        stats=stats, halloffame=hof,
        verbose=False,
    )
    elapsed = time.time() - t0

    best_f1 = hof[0].fitness.values[0]
    print(f"  [Classe '{cls_name}'] Melhor F1 binário (treino): {best_f1:.4f}  | tempo: {elapsed:.0f}s")

    toolbox.unregister("evaluate")
    return hof[0], log, elapsed


print("Função de evolução por classe definida.")

Função de evolução por classe definida.


In [38]:
# ── Executa as 4 evoluções (OvR) ────────────────────────────────────────────
print("=" * 60)
print(f"Iniciando GP One-vs-Rest: {N_CLASSES} classes × {CFG['N_GEN']} gerações × pop {CFG['POP_SIZE']}")
print(f"Treino: {len(X_train):,} instâncias  |  Features: {N_FEATURES}")
print("Estimativa de tempo: ~5–20 min no Colab (GPU não é usada pelo DEAP)")
print("=" * 60)

best_trees = []  # lista com 1 árvore por classe
logs_all   = []
times_all  = []

for k in range(N_CLASSES):
    tree, log, elapsed = evolve_one_class(k, X_train, y_train, toolbox, CFG, seed=SEED + k)
    best_trees.append(tree)
    logs_all.append(log)
    times_all.append(elapsed)

print(f"\nEvolução completa. Tempo total: {sum(times_all):.0f}s ({sum(times_all)/60:.1f} min)")

Iniciando GP One-vs-Rest: 4 classes × 40 gerações × pop 200
Treino: 455,320 instâncias  |  Features: 21
Estimativa de tempo: ~5–20 min no Colab (GPU não é usada pelo DEAP)
  [Classe 'allow'] Evoluindo 200 indivíduos × 40 gerações...
  [Classe 'allow'] Melhor F1 binário (treino): 0.9927  | tempo: 102s
  [Classe 'deny'] Evoluindo 200 indivíduos × 40 gerações...
  [Classe 'deny'] Melhor F1 binário (treino): 0.9154  | tempo: 107s
  [Classe 'drop'] Evoluindo 200 indivíduos × 40 gerações...
  [Classe 'drop'] Melhor F1 binário (treino): 0.4737  | tempo: 117s
  [Classe 'reset-both'] Evoluindo 200 indivíduos × 40 gerações...
  [Classe 'reset-both'] Melhor F1 binário (treino): 0.6729  | tempo: 109s

Evolução completa. Tempo total: 436s (7.3 min)


---
## 6. Weighted Fitness GP — Variante para Classes Minoritárias

Para abordar o desequilíbrio severo de classes (**reset-both = 0,16%** do treino), esta variante
modifica a função de fitness: cada exemplo recebe um `sample_weight` **inversamente proporcional
à frequência de sua classe**. Um exemplo de reset-both pesa ≈ 140× a média das instâncias (≈ 320× um exemplo de allow),
forçando o GP a não ignorar erros nas classes raras.

O conjunto de dados de treino e teste é **idêntico ao do baseline** — apenas a função de fitness muda.

In [39]:
# ── Pesos inversamente proporcionais à frequência de cada classe ─────────────
class_counts = np.bincount(y_train, minlength=N_CLASSES)
class_weights_arr = len(y_train) / (N_CLASSES * class_counts.astype(float))
sample_weights_train = class_weights_arr[y_train]

print("Pesos por classe (quanto cada exemplo conta no fitness):")
for i, (name, w) in enumerate(zip(CLASS_NAMES, class_weights_arr)):
    print(f"  {name:12s}: {w:.2f}×  (n={class_counts[i]:,})")


def make_eval_func_weighted(X_tr, y_tr, class_idx, toolbox, sample_weights):
    """Fitness binário com sample_weight inversamente proporcional à frequência de classe."""
    y_binary = (y_tr == class_idx).astype(np.int32)
    cols = [X_tr[:, j] for j in range(X_tr.shape[1])]

    def evaluate(individual):
        func = toolbox.compile(expr=individual)
        try:
            scores = func(*cols)
            scores = np.broadcast_to(
                np.atleast_1d(np.asarray(scores, dtype=np.float64)),
                (len(y_binary),)
            ).copy()
            preds = (scores > 0).astype(np.int32)
        except Exception:
            return (0.0,)
        f1 = f1_score(y_binary, preds, zero_division=0, sample_weight=sample_weights)
        return (f1,)

    return evaluate


print("make_eval_func_weighted definida.")

Pesos por classe (quanto cada exemplo conta no fitness):
  allow       : 0.44×  (n=259,273)
  deny        : 0.68×  (n=166,496)
  drop        : 3.96×  (n=28,739)
  reset-both  : 140.18×  (n=812)
make_eval_func_weighted definida.


In [40]:
print("=" * 60)
print(f"GP Weighted Fitness: {N_CLASSES} classes × {CFG['N_GEN']} gerações × pop {CFG['POP_SIZE']}")
print(f"Treino: {len(X_train):,} instâncias  |  Features: {N_FEATURES}")
print("=" * 60)

best_trees_weighted = []
logs_weighted       = []
times_weighted      = []

for k in range(N_CLASSES):
    cls_name = CLASS_NAMES[k]
    eval_fn  = make_eval_func_weighted(X_train, y_train, k, toolbox, sample_weights_train)
    toolbox.register("evaluate", eval_fn)

    random.seed(SEED + k)
    np.random.seed(SEED + k)

    pop   = toolbox.population(n=CFG['POP_SIZE'])
    hof   = tools.HallOfFame(CFG['HOF_SIZE'])
    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("max",  np.max)
    stats.register("mean", np.mean)

    t0 = time.time()
    pop, log = algorithms.eaSimple(
        pop, toolbox,
        cxpb=CFG['CXPB'], mutpb=CFG['MUTPB'], ngen=CFG['N_GEN'],
        stats=stats, halloffame=hof, verbose=False,
    )
    elapsed = time.time() - t0

    print(f"  [{cls_name}] Melhor fitness ponderado: {hof[0].fitness.values[0]:.4f} | {elapsed:.0f}s")
    toolbox.unregister("evaluate")
    best_trees_weighted.append(hof[0])
    logs_weighted.append(log)
    times_weighted.append(elapsed)

print(f"\nWeighted — evolução completa em {sum(times_weighted):.0f}s ({sum(times_weighted)/60:.1f} min)")

GP Weighted Fitness: 4 classes × 40 gerações × pop 200
Treino: 455,320 instâncias  |  Features: 21
  [allow] Melhor fitness ponderado: 0.9128 | 135s
  [deny] Melhor fitness ponderado: 0.8021 | 139s
  [drop] Melhor fitness ponderado: 0.8106 | 124s
  [reset-both] Melhor fitness ponderado: 0.9794 | 125s

Weighted — evolução completa em 524s (8.7 min)


---
## 7. Predição & Avaliação de Stream

As predições são geradas para **ambas as variantes** (Baseline e Weighted Fitness) sobre o mesmo
conjunto de teste cronológico (70% do fluxo total). A avaliação cobre três granularidades:

- **Global**: F1 macro e F1 weighted no teste combinado (tabela comparativa)
- **Por período**: DS1 test vs. DS2 test — para medir impacto do concept drift em cada variante
- **50 janelas cronológicas**: para analisar estabilidade e detecção do ponto de drift

In [41]:
# ── Predição vetorizada — Baseline e Weighted Fitness ────────────────────────
def get_raw_scores(X, best_trees, toolbox):
    cols = [X[:, j] for j in range(X.shape[1])]
    scores = np.zeros((len(X), len(best_trees)), dtype=np.float64)
    for k, tree in enumerate(best_trees):
        func = toolbox.compile(expr=tree)
        raw = func(*cols)
        raw = np.broadcast_to(
            np.atleast_1d(np.asarray(raw, dtype=np.float64)),
            (len(X),)
        ).copy()
        scores[:, k] = raw
    return scores


def predict_stream(X, best_trees, toolbox):
    return get_raw_scores(X, best_trees, toolbox).argmax(axis=1).astype(np.int32)


print("Gerando predições — Baseline e Weighted Fitness...")
t0 = time.time()
y_pred_test     = predict_stream(X_test, best_trees,          toolbox)
y_pred_weighted = predict_stream(X_test, best_trees_weighted, toolbox)
print(f"Predições geradas em {time.time()-t0:.1f}s")

# Slices por período (reutilizados nas células seguintes)
y_test_ds1 = y_test[:drift_test_idx]
y_test_ds2 = y_test[drift_test_idx:]

Gerando predições — Baseline e Weighted Fitness...
Predições geradas em 0.2s


In [42]:
# ── F1 global — Baseline vs Weighted Fitness (teste combinado) ───────────────
import pandas as pd

rows = {}
for label, yp in [("Baseline", y_pred_test), ("Weighted", y_pred_weighted)]:
    per_cls = f1_score(y_test, yp, average=None, labels=list(range(N_CLASSES)), zero_division=0)
    rows[label] = {n: round(f, 4) for n, f in zip(CLASS_NAMES, per_cls)}
    rows[label]['macro F1']    = round(f1_score(y_test, yp, average='macro',    zero_division=0), 4)
    rows[label]['weighted F1'] = round(f1_score(y_test, yp, average='weighted', zero_division=0), 4)

df_global = pd.DataFrame(rows).T
print("F1 global — teste combinado\n")
print(df_global.to_string())

# Variáveis de referência para células seguintes (Baseline)
f1_macro_global    = rows["Baseline"]["macro F1"]
f1_weighted_global = rows["Baseline"]["weighted F1"]
f1_per_class       = [rows["Baseline"][n] for n in CLASS_NAMES]

print("\n\nRelatório completo — Baseline:")
print(classification_report(y_test, y_pred_test, target_names=CLASS_NAMES, zero_division=0))
print("Relatório completo — Weighted Fitness:")
print(classification_report(y_test, y_pred_weighted, target_names=CLASS_NAMES, zero_division=0))

F1 global — teste combinado

           allow    deny    drop  reset-both  macro F1  weighted F1
Baseline  0.9913  0.9104  0.0000      0.0000    0.4754       0.8878
Weighted  0.9259  0.8856  0.4734      0.0638    0.5872       0.8767


Relatório completo — Baseline:
              precision    recall  f1-score   support

       allow       0.98      1.00      0.99    559079
        deny       0.85      0.98      0.91    427240
        drop       0.00      0.00      0.00     74448
  reset-both       0.00      0.00      0.00      1648

    accuracy                           0.92   1062415
   macro avg       0.46      0.50      0.48   1062415
weighted avg       0.86      0.92      0.89   1062415

Relatório completo — Weighted Fitness:
              precision    recall  f1-score   support

       allow       0.99      0.87      0.93    559079
        deny       0.93      0.84      0.89    427240
        drop       0.36      0.68      0.47     74448
  reset-both       0.03      0.97      0.06

In [43]:
# ── Avaliação separada por período: DS1 test vs DS2 test ─────────────────────
print("Avaliação por período — DS1 test vs DS2 test\n")

for var_label, yp_full in [("BASELINE", y_pred_test), ("WEIGHTED FITNESS", y_pred_weighted)]:
    print(f"{'─'*55}")
    print(f"  {var_label}")
    print(f"{'─'*55}")
    for period_label, yt, yp in [
        ("DS1 test (70% de DS1)", y_test_ds1, yp_full[:drift_test_idx]),
        ("DS2 test (70% de DS2)", y_test_ds2, yp_full[drift_test_idx:]),
        ("Combinado",             y_test,     yp_full),
    ]:
        print(f"\n  ── {period_label} ({len(yt):,} instâncias) ──")
        print(f"    F1 macro    : {f1_score(yt, yp, average='macro',    zero_division=0):.4f}")
        print(f"    F1 weighted : {f1_score(yt, yp, average='weighted', zero_division=0):.4f}")
        per_cls = f1_score(yt, yp, average=None, labels=list(range(N_CLASSES)), zero_division=0)
        for name, f in zip(CLASS_NAMES, per_cls):
            print(f"      {name:12s}: {f:.4f}")
    print()

Avaliação por período — DS1 test vs DS2 test

───────────────────────────────────────────────────────
  BASELINE
───────────────────────────────────────────────────────

  ── DS1 test (70% de DS1) (701,512 instâncias) ──
    F1 macro    : 0.4764
    F1 weighted : 0.8884
      allow       : 0.9926
      deny        : 0.9128
      drop        : 0.0000
      reset-both  : 0.0000

  ── DS2 test (70% de DS2) (360,903 instâncias) ──
    F1 macro    : 0.4736
    F1 weighted : 0.8865
      allow       : 0.9889
      deny        : 0.9055
      drop        : 0.0000
      reset-both  : 0.0000

  ── Combinado (1,062,415 instâncias) ──
    F1 macro    : 0.4754
    F1 weighted : 0.8878
      allow       : 0.9913
      deny        : 0.9104
      drop        : 0.0000
      reset-both  : 0.0000

───────────────────────────────────────────────────────
  WEIGHTED FITNESS
───────────────────────────────────────────────────────

  ── DS1 test (70% de DS1) (701,512 instâncias) ──
    F1 macro    : 0.5779
  

In [44]:
# ── 50 janelas cronológicas — Baseline & Weighted Fitness ────────────────────
N_WIN = CFG['N_WINDOWS']
n_test = len(y_test)
window_size = n_test // N_WIN

win_f1_macro      = np.zeros(N_WIN)
win_f1_weighted_b = np.zeros(N_WIN)
win_f1_per_class  = np.zeros((N_WIN, N_CLASSES))

win_f1_macro_w     = np.zeros(N_WIN)
win_f1_weighted_w  = np.zeros(N_WIN)
win_f1_per_class_w = np.zeros((N_WIN, N_CLASSES))

for w in range(N_WIN):
    start = w * window_size
    end   = start + window_size if w < N_WIN - 1 else n_test
    yt    = y_test[start:end]

    yp_b = y_pred_test[start:end]
    win_f1_macro[w]      = f1_score(yt, yp_b, average='macro',    zero_division=0)
    win_f1_weighted_b[w] = f1_score(yt, yp_b, average='weighted', zero_division=0)
    win_f1_per_class[w]  = f1_score(yt, yp_b, average=None, labels=list(range(N_CLASSES)), zero_division=0)

    yp_w = y_pred_weighted[start:end]
    win_f1_macro_w[w]     = f1_score(yt, yp_w, average='macro',    zero_division=0)
    win_f1_weighted_w[w]  = f1_score(yt, yp_w, average='weighted', zero_division=0)
    win_f1_per_class_w[w] = f1_score(yt, yp_w, average=None, labels=list(range(N_CLASSES)), zero_division=0)

drift_window = drift_test_idx // window_size

print(f"Avaliação em {N_WIN} janelas concluída.")
print(f"Janela de concept drift (fronteira DS1→DS2): janela {drift_window}")
print(f"\nF1 macro médio — Baseline : {win_f1_macro.mean():.4f} ± {win_f1_macro.std():.4f}")
print(f"F1 macro médio — Weighted : {win_f1_macro_w.mean():.4f} ± {win_f1_macro_w.std():.4f}")

Avaliação em 50 janelas concluída.
Janela de concept drift (fronteira DS1→DS2): janela 33

F1 macro médio — Baseline : 0.4741 ± 0.0088
F1 macro médio — Weighted : 0.5817 ± 0.0255


---
## 8. Figuras para o Relatório

Todas as figuras exibem **Baseline** e **Weighted Fitness** lado a lado para facilitar a comparação.

In [45]:
# ── Figura 1: Curva F1 × janela — Baseline vs Weighted ──────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

x_windows = np.arange(1, N_WIN + 1)

# Painel superior: F1 Macro
axes[0].plot(x_windows, win_f1_macro,   label='Baseline — F1 Macro',  color='#1565C0', lw=1.5)
axes[0].plot(x_windows, win_f1_macro_w, label='Weighted — F1 Macro',  color='#C62828', lw=1.5)
if 0 <= drift_window < N_WIN:
    axes[0].axvline(drift_window + 1, color='#2E7D32', lw=1.5, ls=':',
                    label=f'Concept Drift (jan. {drift_window+1})')
axes[0].set_ylabel('F1 Macro', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(alpha=0.3)

# Painel inferior: F1 Weighted
axes[1].plot(x_windows, win_f1_weighted_b, label='Baseline — F1 Weighted', color='#1565C0', lw=1.5, ls='--')
axes[1].plot(x_windows, win_f1_weighted_w, label='Weighted — F1 Weighted', color='#C62828', lw=1.5, ls='--')
if 0 <= drift_window < N_WIN:
    axes[1].axvline(drift_window + 1, color='#2E7D32', lw=1.5, ls=':')
axes[1].set_ylabel('F1 Weighted', fontsize=11)
axes[1].set_xlabel('Janela de Avaliação', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(alpha=0.3)

fig.suptitle('Desempenho ao Longo do Tempo — 50 Janelas de Avaliação', fontsize=12)
plt.tight_layout()
plt.savefig('figures/fig_f1_janelas.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: fig_f1_janelas.png")

Figura salva: fig_f1_janelas.png


In [46]:
# ── Figura 2: F1 por classe ao longo das janelas — Baseline vs Weighted ──────
colors_cls = ['#1565C0', '#C62828', '#FF6F00', '#6A1B9A']

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

for k, (cls, c) in enumerate(zip(CLASS_NAMES, colors_cls)):
    axes[0].plot(x_windows, win_f1_per_class[:, k],   label=cls, color=c, lw=1.3)
    axes[1].plot(x_windows, win_f1_per_class_w[:, k], label=cls, color=c, lw=1.3)

for ax, title in zip(axes, ['Baseline', 'Weighted Fitness']):
    if 0 <= drift_window < N_WIN:
        ax.axvline(drift_window + 1, color='#2E7D32', lw=1.5, ls=':', label='Concept Drift')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('F1', fontsize=10)
    ax.legend(fontsize=9, ncol=2)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(alpha=0.3)

axes[1].set_xlabel('Janela de Avaliação', fontsize=11)
fig.suptitle('F1 por Classe ao Longo do Tempo — Baseline vs Weighted Fitness', fontsize=12)
plt.tight_layout()
plt.savefig('figures/fig_f1_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: fig_f1_por_classe.png")

Figura salva: fig_f1_por_classe.png


In [47]:
# ── Figura 3: Matrizes de confusão normalizadas — Baseline e Weighted ─────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, yp, title in zip(
    axes,
    [y_pred_test, y_pred_weighted],
    ['Baseline', 'Weighted Fitness']
):
    cm = confusion_matrix(y_test, yp, labels=list(range(N_CLASSES)))
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(N_CLASSES))
    ax.set_yticks(range(N_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel('Predito')
    ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusão — {title}', fontsize=11)

    for i in range(N_CLASSES):
        for j in range(N_CLASSES):
            color = 'white' if cm_norm[i, j] > 0.55 else 'black'
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                    color=color, fontsize=9)

plt.tight_layout()
plt.savefig('figures/fig_confusao.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: fig_confusao.png")

Figura salva: fig_confusao.png


In [48]:
# ── Figura 4: Convergência do fitness — Baseline vs Weighted (2 × 4) ─────────
colors_cls = ['#1565C0', '#C62828', '#FF6F00', '#6A1B9A']
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex=True, sharey=True)

for row_idx, (logs, row_title) in enumerate([(logs_all, 'Baseline'), (logs_weighted, 'Weighted Fitness')]):
    for k, (cls, c, log) in enumerate(zip(CLASS_NAMES, colors_cls, logs)):
        ax = axes[row_idx][k]
        gens  = [entry['gen']  for entry in log]
        maxes = [entry['max']  for entry in log]
        means = [entry['mean'] for entry in log]
        ax.plot(gens, maxes, color=c, lw=1.5, label='Melhor')
        ax.plot(gens, means, color=c, lw=1,   ls='--', alpha=0.7, label='Média')
        ax.set_title(f"[{row_title}]\n'{cls}'", fontsize=8)
        if row_idx == 0 and k == 0:
            ax.legend(fontsize=7)
        ax.set_ylim(-0.05, 1.05)
        ax.grid(alpha=0.3)

for ax in axes[:, 0]:
    ax.set_ylabel('Fitness (F1 binário / ponderado)')
for ax in axes[1]:
    ax.set_xlabel('Geração')

fig.suptitle('Convergência do Fitness GP por Classe — Baseline vs Weighted Fitness', fontsize=12)
plt.tight_layout()
plt.savefig('figures/fig_convergencia.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: fig_convergencia.png")

Figura salva: fig_convergencia.png


In [49]:
# ── Figura 5: Complexidade e expressões das árvores — Baseline vs Weighted ────
print("Melhores expressões — BASELINE:")
for k, (cls, t) in enumerate(zip(CLASS_NAMES, best_trees)):
    expr = str(t)
    print(f"  [{cls}] altura={t.height}  nós={len(t)}")
    print(f"    {expr[:120]}{'...' if len(expr) > 120 else ''}")

print("\nMelhores expressões — WEIGHTED FITNESS:")
for k, (cls, t) in enumerate(zip(CLASS_NAMES, best_trees_weighted)):
    expr = str(t)
    print(f"  [{cls}] altura={t.height}  nós={len(t)}")
    print(f"    {expr[:120]}{'...' if len(expr) > 120 else ''}")

# Gráfico de barras: número de nós por variante e classe
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(N_CLASSES)
width = 0.35
sizes_base = [len(t) for t in best_trees]
sizes_wt   = [len(t) for t in best_trees_weighted]

bars1 = ax.bar(x - width/2, sizes_base, width, label='Baseline',         color='#1565C0', alpha=0.85)
bars2 = ax.bar(x + width/2, sizes_wt,   width, label='Weighted Fitness', color='#C62828', alpha=0.85)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
            str(int(h)), ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('Número de nós')
ax.set_title('Complexidade das Árvores Evoluídas', fontsize=12)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig_arvores.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: fig_arvores.png")

Melhores expressões — BASELINE:
  [allow] altura=6  nós=15
    neg(add(min2(x0, div(sub(x8, x13), max2(abs(x16), x20))), max2(x1, x8)))
  [deny] altura=4  nós=15
    add(add(x0, add(add(x1, x0), x0)), add(add(x1, x0), mul(x3, x17)))
  [drop] altura=6  nós=29
    min2(min2(neg(min2(mul(x17, x16), sub(x3, x2))), min2(x3, min2(min2(neg(x15), min2(x0, x1)), neg(x5)))), neg(neg(mul(neg...
  [reset-both] altura=6  nós=27
    min2(add(div(x18, neg(x7)), sub(x18, max2(max2(add(x7, x7), div(x16, x4)), add(abs(x10), x16)))), sub(div(x16, x4), min2...

Melhores expressões — WEIGHTED FITNESS:
  [allow] altura=6  nós=34
    sub(x12, div(mul(min2(x14, neg(sub(x0, x18))), neg(x9)), max2(div(mul(min2(x14, x11), neg(x9)), max2(add(x14, x3), x12))...
  [deny] altura=5  nós=20
    min2(max2(div(max2(sub(x13, x1), sub(x18, x16)), max2(neg(x5), x0)), sub(x18, mul(x15, x3))), x0)
  [drop] altura=6  nós=19
    add(min2(x0, sub(sub(add(mul(x18, x9), x20), max2(x7, x10)), mul(x18, add(0.28067006386626314, x1))

In [50]:
# ── Figura 6: Comparação F1 por classe — Baseline vs Weighted (barras) ───────
rows_comp = {}
for label, yp in [("Baseline", y_pred_test), ("Weighted", y_pred_weighted)]:
    per_cls = f1_score(y_test, yp, average=None, labels=list(range(N_CLASSES)), zero_division=0)
    rows_comp[label] = {n: round(f, 4) for n, f in zip(CLASS_NAMES, per_cls)}
    rows_comp[label]['Macro F1'] = round(f1_score(y_test, yp, average='macro', zero_division=0), 4)

cols_plot = CLASS_NAMES + ['Macro F1']
x_c = np.arange(len(cols_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
vals_base = [rows_comp['Baseline'][c] for c in cols_plot]
vals_wt   = [rows_comp['Weighted'][c]  for c in cols_plot]

bars1 = ax.bar(x_c - width/2, vals_base, width, label='Baseline',         color='#1565C0', alpha=0.85)
bars2 = ax.bar(x_c + width/2, vals_wt,   width, label='Weighted Fitness', color='#C62828', alpha=0.85)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
            f'{h:.3f}', ha='center', va='bottom', fontsize=7.5, rotation=45)

ax.set_xticks(x_c)
ax.set_xticklabels(cols_plot, fontsize=10)
ax.set_ylabel('F1', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_title('Baseline vs Weighted Fitness — F1 por Classe e F1 Macro (teste combinado)', fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/fig_comparacao.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva: fig_comparacao.png")

Figura salva: fig_comparacao.png


---
## 9. Sumário de Resultados

In [51]:
print("=" * 65)
print("SUMÁRIO FINAL — Baseline vs Weighted Fitness")
print("=" * 65)
print(f"Algoritmo : Programação Genética (DEAP) – One-vs-Rest")
print(f"Datasets  : DS1 ({CFG['FILE_DS1']})")
print(f"            DS2 ({CFG['FILE_DS2']})")
print(f"Fluxo total: {n_total:,}  |  Treino (30%): {n_train:,}  |  Teste (70%): {len(y_test):,}")
print(f"Features  : {N_FEATURES}")
print()
print(f"GP Config : pop={CFG['POP_SIZE']}  gen={CFG['N_GEN']}  "
      f"cxpb={CFG['CXPB']}  mutpb={CFG['MUTPB']}  max_height={CFG['MAX_HEIGHT']}")
print(f"Tempo Baseline : {sum(times_all):.0f}s ({sum(times_all)/60:.1f} min)")
print(f"Tempo Weighted : {sum(times_weighted):.0f}s ({sum(times_weighted)/60:.1f} min)")
print()

# ── Tabela comparativa ──────────────────────────────────────────────────────
import pandas as pd
rows_s = {}
for label, yp in [("Baseline", y_pred_test), ("Weighted", y_pred_weighted)]:
    per_cls = f1_score(y_test, yp, average=None, labels=list(range(N_CLASSES)), zero_division=0)
    rows_s[label] = {n: f'{f:.4f}' for n, f in zip(CLASS_NAMES, per_cls)}
    rows_s[label]['macro F1']    = f'{f1_score(y_test, yp, average="macro",    zero_division=0):.4f}'
    rows_s[label]['weighted F1'] = f'{f1_score(y_test, yp, average="weighted", zero_division=0):.4f}'

print("DESEMPENHO NO TESTE COMBINADO:")
print(pd.DataFrame(rows_s).T.to_string())
print()

# ── F1 macro por período ────────────────────────────────────────────────────
print("F1 MACRO POR PERÍODO (Concept Drift):")
for label, yp in [("Baseline", y_pred_test), ("Weighted", y_pred_weighted)]:
    f1_ds1 = f1_score(y_test_ds1, yp[:drift_test_idx], average='macro', zero_division=0)
    f1_ds2 = f1_score(y_test_ds2, yp[drift_test_idx:], average='macro', zero_division=0)
    print(f"  {label:<10}: DS1={f1_ds1:.4f}  DS2={f1_ds2:.4f}  "
          f"(ganho DS2-DS1={f1_ds2-f1_ds1:+.4f})")

print()
# ── Estatísticas das 50 janelas ─────────────────────────────────────────────
print("F1 MACRO — 50 JANELAS:")
print(f"  Baseline: média={win_f1_macro.mean():.4f}  desvio={win_f1_macro.std():.4f}  "
      f"[min={win_f1_macro.min():.4f}  max={win_f1_macro.max():.4f}]")
print(f"  Weighted: média={win_f1_macro_w.mean():.4f}  desvio={win_f1_macro_w.std():.4f}  "
      f"[min={win_f1_macro_w.min():.4f}  max={win_f1_macro_w.max():.4f}]")

print()
print("FIGURAS GERADAS:")
figs = ['fig_eda_distribuicao.png', 'fig_f1_janelas.png', 'fig_f1_por_classe.png',
        'fig_confusao.png', 'fig_convergencia.png', 'fig_arvores.png', 'fig_comparacao.png']
for fig_name in figs:
    status = 'OK' if os.path.exists(fig_name) else 'FALTANDO'
    print(f"  {status}  {fig_name}")
print("=" * 65)

SUMÁRIO FINAL — Baseline vs Weighted Fitness
Algoritmo : Programação Genética (DEAP) – One-vs-Rest
Datasets  : DS1 (Log-Completo-Todos-Atributos-Importantes.csv)
            DS2 (log-completo-Todos-Atributos-Importantes-fase-online-2.csv)
Fluxo total: 1,517,735  |  Treino (30%): 455,320  |  Teste (70%): 1,062,415
Features  : 21

GP Config : pop=200  gen=40  cxpb=0.7  mutpb=0.3  max_height=6
Tempo Baseline : 436s (7.3 min)
Tempo Weighted : 524s (8.7 min)

DESEMPENHO NO TESTE COMBINADO:
           allow    deny    drop reset-both macro F1 weighted F1
Baseline  0.9913  0.9104  0.0000     0.0000   0.4754      0.8878
Weighted  0.9259  0.8856  0.4734     0.0638   0.5872      0.8767

F1 MACRO POR PERÍODO (Concept Drift):
  Baseline  : DS1=0.4764  DS2=0.4736  (ganho DS2-DS1=-0.0028)
  Weighted  : DS1=0.5779  DS2=0.6059  (ganho DS2-DS1=+0.0280)

F1 MACRO — 50 JANELAS:
  Baseline: média=0.4741  desvio=0.0088  [min=0.4497  max=0.4865]
  Weighted: média=0.5817  desvio=0.0255  [min=0.5361  max=0.63